# Model Evaluation & Error Analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torchvision.utils import make_grid

from sklearn.metrics import accuracy_score, classification_report
from omegaconf import OmegaConf
from src.training.checkpoint import load_checkpoint
from src.viz.confusion import plot_confusion_matrix
from src.viz.roc_pr import plot_roc_curves
from src.viz.gradcam import generate_gradcam

%matplotlib inline

In [ ]:
ckpt = load_checkpoint("../checkpoints/last.pt")
print("=" * 50)
print("CHECKPOINT METADATA")
print("=" * 50)
print(f"Epoch:              {ckpt.get('epoch', 'N/A')}")
print(f"Global step:        {ckpt.get('global_step', 'N/A')}")
print(f"Timestamp:          {ckpt.get('timestamp', 'N/A')}")
print(f"PyTorch version:    {ckpt.get('pytorch_version', 'N/A')}")
print(f"CUDA version:       {ckpt.get('cuda_version', 'N/A')}")
print(f"Git hash:           {ckpt.get('git_hash', 'N/A')}")
print(f"Device:             {ckpt.get('device', 'N/A')}")
print(f"Seed:               {ckpt.get('seed', 'N/A')}")
if 'best_metric' in ckpt and ckpt['best_metric']:
    best_k, best_v = list(ckpt['best_metric'].items())[0]
    print(f"Best metric:        {best_k} = {best_v:.6f}")
print(f"Config keys:        {list(ckpt.get('config', {}).keys())[:10]}")
print("=" * 50)

In [ ]:
metrics_history = ckpt.get('metrics_history', [])
if metrics_history:
    train_losses = [m.get('train/loss') or m.get('train_ce') for m in metrics_history]
    val_losses = [m.get('val/loss') or m.get('val_ce') for m in metrics_history]
    train_acc = [m.get('train/accuracy') for m in metrics_history]
    val_acc = [m.get('val/accuracy') for m in metrics_history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(train_losses) + 1)
    axes[0].plot(epochs, train_losses, 'b-', label='Train Loss', linewidth=2)
    axes[0].plot(epochs, val_losses, 'r-', label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, train_acc, 'b-', label='Train Accuracy', linewidth=2)
    axes[1].plot(epochs, val_acc, 'r-', label='Val Accuracy', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training & Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No metrics history found in checkpoint.")

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self, num_classes=10, backbone_dim=512):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.head = nn.Linear(64, num_classes)

    def forward(self, x):
        feats = self.backbone(x).flatten(1)
        return {"logits": self.head(feats)}

config = ckpt.get('config', {})
num_classes = config.get('experiment', {}).get('num_classes', 10)
model = SimpleModel(num_classes=num_classes)

if 'model_state' in ckpt:
    model_state = ckpt['model_state']
    model_state = {k.replace('model.', ''): v for k, v in model_state.items()}
    try:
        model.load_state_dict(model_state, strict=False)
        print("Model weights loaded successfully.")
    except Exception as e:
        print(f"Could not load model weights: {e}")
model.eval()

test_images = torch.randn(100, 3, 224, 224)
test_labels = torch.randint(0, num_classes, (100,))

all_preds = []
all_labels = []
all_probs = []
with torch.no_grad():
    for i in range(0, 100, 16):
        batch_imgs = test_images[i:i+16]
        batch_lbls = test_labels[i:i+16]
        outputs = model(batch_imgs)
        logits = outputs['logits']
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.tolist())
        all_labels.extend(batch_lbls.tolist())
        all_probs.append(probs)

all_probs = torch.cat(all_probs, dim=0).numpy()
acc = accuracy_score(all_labels, all_preds)
print(f"Test accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=3))

In [ ]:
class_names = [f"Class {i}" for i in range(num_classes)]
fig = plot_confusion_matrix(
    all_labels, all_preds, class_names,
    normalize=True, figsize=(10, 8)
)
plt.show()

In [ ]:
fig = plot_roc_curves(
    all_labels, all_probs, class_names,
    figsize=(10, 8)
)
plt.show()

In [ ]:
all_preds_np = np.array(all_preds)
all_labels_np = np.array(all_labels)
correct_mask = all_preds_np == all_labels_np
correct_indices = np.where(correct_mask)[0]

n_show = min(16, len(correct_indices))
if n_show > 0:
    selected = np.random.choice(correct_indices, size=n_show, replace=False)
    correct_imgs = test_images[selected]
    correct_labels = test_labels[selected]

    correct_imgs = (correct_imgs - correct_imgs.min()) / (correct_imgs.max() - correct_imgs.min() + 1e-8)
    grid = make_grid(correct_imgs, nrow=4, normalize=False)
    grid = grid.permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(grid)
    titles = [f"True: {correct_labels[i].item()}" for i in range(n_show)]
    cols = 4
    rows = (n_show + cols - 1) // cols
    for idx, title in enumerate(titles):
        r, c = divmod(idx, cols)
        x = c / cols + 0.5 / cols
        y = 1.0 - (r + 0.92) / rows
        ax.text(x, y, title, transform=ax.transAxes, fontsize=8, ha='center', va='top',
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    ax.axis('off')
    ax.set_title('Correct Predictions')
    plt.tight_layout()
    plt.show()
else:
    print("No correct predictions to display.")

In [ ]:
wrong_mask = all_preds_np != all_labels_np
wrong_indices = np.where(wrong_mask)[0]

n_show = min(16, len(wrong_indices))
if n_show > 0:
    selected = np.random.choice(wrong_indices, size=n_show, replace=False)
    wrong_imgs = test_images[selected]
    wrong_true = all_labels_np[selected]
    wrong_pred = all_preds_np[selected]

    wrong_imgs = (wrong_imgs - wrong_imgs.min()) / (wrong_imgs.max() - wrong_imgs.min() + 1e-8)
    grid = make_grid(wrong_imgs, nrow=4, normalize=False)
    grid = grid.permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(grid)
    titles = [f"T:{wrong_true[i]} P:{wrong_pred[i]}" for i in range(n_show)]
    cols = 4
    rows = (n_show + cols - 1) // cols
    for idx, title in enumerate(titles):
        r, c = divmod(idx, cols)
        x = c / cols + 0.5 / cols
        y = 1.0 - (r + 0.92) / rows
        ax.text(x, y, title, transform=ax.transAxes, fontsize=6, ha='center', va='top',
                color='red', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    ax.axis('off')
    ax.set_title('Misclassified Examples')
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified examples.")

In [ ]:
sample_input = test_images[0:1]
sample_label = test_labels[0].item()
try:
    target_layer = model.backbone[-2]
    overlay = generate_gradcam(
        model, sample_input, target_layer,
        class_idx=sample_label,
        save_path="./gradcam_sample.png"
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    original = sample_input[0].permute(1, 2, 0).numpy()
    original = (original - original.min()) / (original.max() - original.min() + 1e-8)
    axes[0].imshow(original)
    axes[0].set_title(f'Original (Class {sample_label})')
    axes[0].axis('off')

    axes[1].imshow(overlay)
    axes[1].set_title(f'Grad-CAM Overlay')
    axes[1].axis('off')

    cam_raw = generate_gradcam(
        model, sample_input, target_layer,
        class_idx=sample_label,
        alpha=1.0
    )
    axes[2].imshow(cam_raw)
    axes[2].set_title(f'Grad-CAM Heatmap')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Grad-CAM generation skipped: {e}")